# IEEE-CID Fraud Detection — Data Preprocessing 
Full preprocessing pipeline for the deep learning model.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import pickle
import os
import zipfile

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load & Merge

In [2]:
raw_data_dir = os.path.join('..', 'Data', 'raw_data')

files = [
    ('train_transaction.zip', 'train_transaction.csv'),
    ('train_identity.zip', 'train_identity.csv')
]

for zip_name, csv_name in files:
    zip_path = os.path.join(raw_data_dir, zip_name)
    csv_path = os.path.join(raw_data_dir, csv_name)

    if not os.path.exists(csv_path):
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(raw_data_dir)
            print(f'Extracted {zip_name}')
        else:
            raise FileNotFoundError(f'Missing both {csv_name} and {zip_name}')
    else:
        print(f'{csv_name} already exists')

train_transaction.csv already exists
train_identity.csv already exists


In [3]:
transaction = pd.read_csv('../Data/raw_data/train_transaction.csv')
identity    = pd.read_csv('../Data/raw_data/train_identity.csv')

df = transaction.merge(identity, on='TransactionID', how='left')

print(f'Transaction rows : {len(transaction):,}')
print(f'Identity rows    : {len(identity):,}')
print(f'Merged rows      : {len(df):,}')
print(f'Merged columns   : {df.shape[1]}')

df.head()

Transaction rows : 590,540
Identity rows    : 144,233
Merged rows      : 590,540
Merged columns   : 434


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## 3. Define Column Groups

In [4]:
target    = 'isFraud'
drop_cols = ['TransactionID']

# Confirmed categorical columns per IEEE-CID dataset documentation
# Transaction: ProductCD, card1-card6, addr1, addr2, P_emaildomain, R_emaildomain, M1-M9
# Identity:    DeviceType, DeviceInfo, id_12-id_38
known_cat_cols = [
    'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo',
    'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19',
    'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27',
    'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35',
    'id_36', 'id_37', 'id_38'
]

cat_cols = []
num_cols = []

for col in df.columns:
    if col in ([target] + drop_cols):
        continue
    if df[col].isna().all():
        continue  # entirely null — will be dropped in cleaning
    if col in known_cat_cols:
        cat_cols.append(col)
    else:
        first_valid = df[col].dropna().iloc[0]
        try:
            float(first_valid)
            num_cols.append(col)
        except (ValueError, TypeError):
            cat_cols.append(col)  # unexpected string column — treat as categorical

print(f'Categorical columns : {len(cat_cols)}')
print(f'Numerical columns   : {len(num_cols)}')
print(f'Target              : {target}')
print(f'\nCategorical: {cat_cols}')

Categorical columns : 49
Numerical columns   : 383
Target              : isFraud

Categorical: ['ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


## 4. EDA

In [5]:
print('All columns in the dataset:')
print(df.columns.tolist())

All columns in the dataset:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V

In [6]:
fraud_percentage = df['isFraud'].mean() * 100
print(f'Percentage of fraudulent transactions: {fraud_percentage:.4f}%')

fraud_counts = df['isFraud'].value_counts()
print('Number of transactions by class:')
print(fraud_counts)

Percentage of fraudulent transactions: 3.4990%
Number of transactions by class:
isFraud
0    569877
1     20663
Name: count, dtype: int64


In [7]:
# Standardise missing value representations
missing_values = ['', 'NA', 'NaN', 'None']
df.replace(missing_values, np.nan, inplace=True)

# Missing value summary
nan_summary = pd.DataFrame({
    'nan_count':      df.isnull().sum(),
    'nan_percentage': df.isnull().mean() * 100
})
nan_summary = nan_summary[nan_summary['nan_count'] > 0].sort_values('nan_percentage', ascending=False)
print(nan_summary)

       nan_count  nan_percentage
id_24     585793       99.196159
id_25     585408       99.130965
id_07     585385       99.127070
id_08     585385       99.127070
id_21     585381       99.126393
...          ...             ...
V309          12        0.002032
V312          12        0.002032
V311          12        0.002032
V310          12        0.002032
V316          12        0.002032

[414 rows x 2 columns]


In [8]:
# Inspect high-missing numeric columns and their correlations
tbd_columns = nan_summary[nan_summary['nan_percentage'] > 90].index.tolist()
print(f'Columns with >90% missing: {len(tbd_columns)}')

numeric_tbd_columns = df[tbd_columns].select_dtypes(include='number').columns.tolist()
categorical_tbd_columns = df[tbd_columns].select_dtypes(include=['object', 'category']).columns.tolist()

print(f'  Numeric   : {len(numeric_tbd_columns)}')
print(f'  Categorical: {len(categorical_tbd_columns)}')

correlation_matrix = df[numeric_tbd_columns].corr()
print('\nCorrelation matrix for high-missing numeric columns:')
print(correlation_matrix)

Columns with >90% missing: 12
  Numeric   : 10
  Categorical: 2

Correlation matrix for high-missing numeric columns:
          id_24     id_25     id_07     id_08     id_21     id_26     id_22  \
id_24  1.000000 -0.030902 -0.070752 -0.011849  0.220933  0.086002  0.014067   
id_25 -0.030902  1.000000  0.037649 -0.003628 -0.147694  0.011508  0.182643   
id_07 -0.070752  0.037649  1.000000 -0.094086 -0.188250 -0.131638 -0.277448   
id_08 -0.011849 -0.003628 -0.094086  1.000000  0.085533  0.037212  0.143301   
id_21  0.220933 -0.147694 -0.188250  0.085533  1.000000  0.053721  0.070591   
id_26  0.086002  0.011508 -0.131638  0.037212  0.053721  1.000000  0.281324   
id_22  0.014067  0.182643 -0.277448  0.143301  0.070591  0.281324  1.000000   
dist2  0.044682 -0.042122 -0.044094  0.034848  0.101177  0.125198  0.071200   
D7     0.138680 -0.034202  0.036789  0.109378 -0.023208  0.179871  0.070430   
id_18  0.014321  0.108161 -0.161544 -0.061456  0.202953 -0.048799  0.130109   

          di

C:\Users\65938\AppData\Local\Temp\ipykernel_21176\666060271.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_tbd_columns = df[tbd_columns].select_dtypes(include=['object', 'category']).columns.tolist()


## 5. Clean Data
- Drop columns with >90% missing as they are too sparse to be useful
- Do median imputation for remaining numerical NaNs as neural networks cannot handle NaN values
- Fill categorical NaNs with 'missing'
- Winsorisation of numerical outliers at 1st/99th percentile as neural networks are sensitive to extreme values
- Drop TransactionID

In [9]:
# Drop columns with high number of missing values
missing_rate = df[num_cols + cat_cols].isnull().mean()
high_missing = missing_rate[missing_rate > 0.9].index.tolist()

print(f'Dropping {len(high_missing)} columns with >90% missing values')
df.drop(columns=high_missing, inplace=True)
cat_cols = [c for c in cat_cols if c not in high_missing]
num_cols = [c for c in num_cols if c not in high_missing]

# Impute numerical NaNs with median 
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Impute categorical NaNs with 'missing' 
for col in cat_cols:
    df[col] = df[col].astype(str).replace('nan', 'missing')

# Winsorise numerical outliers at 1st / 99th percentile 
for col in num_cols:
    low  = df[col].quantile(0.01)
    high = df[col].quantile(0.99)
    df[col] = df[col].clip(low, high)

# Drop ID column 
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

print(f'Shape after cleaning : {df.shape}')
print(f'Remaining nulls      : {df.isnull().sum().sum()}')

Dropping 12 columns with >90% missing values
Shape after cleaning : (590540, 421)
Remaining nulls      : 13159948


## 6. Train / Val / Test Split (80 / 5 / 15)
Stratification done to ensure all three splits keep the same ~3.5% fraud rate.

In [10]:
# Step 1: split off 15% test
train_val, test = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df[target]
)

# Step 2: split remaining 85% → 80% train, 5% val
# 5% of total = 5/85 ≈ 0.0588 of train_val
train, val = train_test_split(
    train_val, test_size=0.0588, random_state=42, stratify=train_val[target]
)

# Reset indices for clean indexing
train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

print(f'Train      : {len(train):,} rows ({len(train)/len(df):.1%}) | fraud rate: {train[target].mean():.3%}')
print(f'Validation : {len(val):,} rows ({len(val)/len(df):.1%})  | fraud rate: {val[target].mean():.3%}')
print(f'Test       : {len(test):,} rows ({len(test)/len(df):.1%}) | fraud rate: {test[target].mean():.3%}')

Train      : 472,443 rows (80.0%) | fraud rate: 3.499%
Validation : 29,516 rows (5.0%)  | fraud rate: 3.500%
Test       : 88,581 rows (15.0%) | fraud rate: 3.498%


## 7. Feature Engineering

### Why val and test still receive new columns
The model is trained expecting a fixed set of input columns (e.g. card1_freq, card1_amt_mean).
If val and test don't have those same columns, the model cannot make predictions on them.

- COMPUTE every statistic and mapping from train only. val/test never contribute to any calculation.
- STAMP those train-derived values onto val and test as a lookup so no new information is learned from val/test

### Features added
- Time-based: hour of day, day-of-week proxy, weekend flag 
- Frequency: how often each card / email domain appears 
- Aggregation: per-card transaction amount mean/std/count, deviation from card mean 
- Transaction amount: log transform, cents (decimal part)
- Interaction: card+address and card+email combination frequency 
- M-column summary: count of T / F / null across M1–M9 

In [11]:
def apply_feature_engineering(train_df, val_df, test_df):

    # Time-based features
    for split in [train_df, val_df, test_df]:
        split['Transaction_hour']    = (split['TransactionDT'] // 3600) % 24
        split['Transaction_day']     = (split['TransactionDT'] // 86400) % 7
        split['Transaction_weekend'] = (split['Transaction_day'] >= 5).astype(int)

    # Card frequency 
    card_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']
    for col in card_cols:
        if col not in train_df.columns:
            continue
        freq_map = train_df[col].value_counts().to_dict()  
        for split in [train_df, val_df, test_df]:          
            split[f'{col}_freq'] = split[col].map(freq_map).fillna(0)

    # Email domain frequency 
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col not in train_df.columns:
            continue
        freq_map = train_df[col].value_counts().to_dict()  
        for split in [train_df, val_df, test_df]:          
            split[f'{col}_freq'] = split[col].map(freq_map).fillna(0)

    # Per-card transaction amount stats 
    card_num_cols = ['card1', 'card2', 'card3', 'card5']
    for col in card_num_cols:
        if col not in train_df.columns:
            continue

        stats = train_df.groupby(col)['TransactionAmt'].agg(['mean', 'std', 'count'])
        stats.columns = [f'{col}_amt_mean', f'{col}_amt_std', f'{col}_count']

        for split_name, split_ref in [('train', train_df), ('val', val_df), ('test', test_df)]:
            merged = split_ref.merge(stats, on=col, how='left')
            merged[f'{col}_amt_diff_from_mean'] = merged['TransactionAmt'] - merged[f'{col}_amt_mean']
            merged[f'{col}_amt_ratio_to_mean']  = merged['TransactionAmt'] / (merged[f'{col}_amt_mean'] + 1e-9)
            # Fill 0 for cards in val/test not seen in train
            for feat in [f'{col}_amt_mean', f'{col}_amt_std', f'{col}_count',
                         f'{col}_amt_diff_from_mean', f'{col}_amt_ratio_to_mean']:
                merged[feat] = merged[feat].fillna(0)
            if split_name == 'train': train_df = merged
            elif split_name == 'val': val_df   = merged
            else:                     test_df  = merged

    # Transaction amount features 
    for split in [train_df, val_df, test_df]:
        split['TransactionAmt_log']   = np.log1p(split['TransactionAmt'])
        split['TransactionAmt_cents'] = (split['TransactionAmt'] % 1).round(2)

    # Interaction features: card + address / email
    for col in card_cols:
        if col not in train_df.columns:
            continue
        if 'addr1' in train_df.columns:
            combo_train = train_df[col].astype(str) + '_' + train_df['addr1'].astype(str)
            freq_map = combo_train.value_counts().to_dict()  
            for split in [train_df, val_df, test_df]:        
                combo = split[col].astype(str) + '_' + split['addr1'].astype(str)
                split[f'{col}_addr1_freq'] = combo.map(freq_map).fillna(0)

        if 'P_emaildomain' in train_df.columns:
            combo_train = train_df[col].astype(str) + '_' + train_df['P_emaildomain'].astype(str)
            freq_map = combo_train.value_counts().to_dict()  # FIT on train
            for split in [train_df, val_df, test_df]:        # APPLY to all
                combo = split[col].astype(str) + '_' + split['P_emaildomain'].astype(str)
                split[f'{col}_email_freq'] = combo.map(freq_map).fillna(0)

    # M-column summary features 
    m_cols = [c for c in train_df.columns if c.startswith('M') and c[1:].isdigit()]
    if m_cols:
        for split in [train_df, val_df, test_df]:
            split['M_true_count']  = (split[m_cols] == 'T').sum(axis=1)
            split['M_false_count'] = (split[m_cols] == 'F').sum(axis=1)
            split['M_null_count']  = split[m_cols].isnull().sum(axis=1)

    return train_df, val_df, test_df


train, val, test = apply_feature_engineering(train, val, test)

print(f'Train shape after feature engineering : {train.shape}')
print(f'Val shape after feature engineering   : {val.shape}')
print(f'Test shape after feature engineering  : {test.shape}')

C:\Users\65938\AppData\Local\Temp\ipykernel_21176\3928076865.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  split['Transaction_hour']    = (split['TransactionDT'] // 3600) % 24
C:\Users\65938\AppData\Local\Temp\ipykernel_21176\3928076865.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  split['Transaction_day']     = (split['TransactionDT'] // 86400) % 7
C:\Users\65938\AppData\Local\Temp\ipykernel_21176\3928076865.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inser

Train shape after feature engineering : (472443, 469)
Val shape after feature engineering   : (29516, 469)
Test shape after feature engineering  : (88581, 469)


C:\Users\65938\AppData\Local\Temp\ipykernel_21176\3928076865.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  split['M_false_count'] = (split[m_cols] == 'F').sum(axis=1)
C:\Users\65938\AppData\Local\Temp\ipykernel_21176\3928076865.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  split['M_null_count']  = split[m_cols].isnull().sum(axis=1)


In [12]:
# Refresh both lists against what actually exists in train after feature engineering.
cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in train.columns if c not in set(cat_cols + [target])]

# Impute any NaNs introduced by merges in val/test (unseen card values)
for col in num_cols:
    for split in [train, val, test]:
        if split[col].isnull().any():
            split[col] = split[col].fillna(split[col].median())

print(f'Updated — Categorical: {len(cat_cols)}, Numerical: {len(num_cols)}')

Updated — Categorical: 41, Numerical: 427


## 8. Encode Categorical Features
- card4 and card6 (e.g. visa/mastercard) are one-hot encoded as they have low cardinality, no meaningful order
- All other categoricals are label encoded to integers for use with nn.Embedding layers
- Unseen categories in val/test are mapped to an out-of-vocabulary (OOV) index

In [13]:
encoders = {}

for col in cat_cols:
    if col not in train.columns:
        continue

    le = LabelEncoder()
    le.fit(train[col])

    class_map = {c: i for i, c in enumerate(le.classes_)}
    oov_index = len(le.classes_)

    for split in [train, val, test]:
        split[col] = split[col].map(class_map).fillna(oov_index).astype(int)

    encoders[col] = le

vocab_sizes = {col: len(le.classes_) + 1 for col, le in encoders.items()}

print(f'Label encoded {len(encoders)} categorical columns.')
print('\nVocabulary sizes (for Embedding layers):')
for col, size in vocab_sizes.items():
    print(f'  {col}: {size}')

Label encoded 41 categorical columns.

Vocabulary sizes (for Embedding layers):
  ProductCD: 6
  card1: 12819
  card2: 502
  card3: 115
  card4: 6
  card5: 116
  card6: 6
  addr1: 307
  addr2: 71
  P_emaildomain: 61
  R_emaildomain: 62
  M1: 4
  M2: 4
  M3: 4
  M4: 5
  M5: 4
  M6: 4
  M7: 4
  M8: 4
  M9: 4
  id_12: 4
  id_13: 55
  id_14: 26
  id_15: 5
  id_16: 4
  id_17: 103
  id_19: 517
  id_20: 386
  id_28: 4
  id_29: 4
  id_30: 77
  id_31: 128
  id_32: 6
  id_33: 233
  id_34: 6
  id_35: 4
  id_36: 4
  id_37: 4
  id_38: 4
  DeviceType: 4
  DeviceInfo: 1680


## 9. Scale Numerical Features
StandardScaler fit on train only, applied to val and test. Necessary so that the neural network eventually converges.

In [14]:
non_numeric = set([target] + list(encoders.keys()))
num_cols_to_scale = [c for c in train.columns if c not in non_numeric]

scaler = StandardScaler()
train[num_cols_to_scale] = scaler.fit_transform(train[num_cols_to_scale])
val[num_cols_to_scale]   = scaler.transform(val[num_cols_to_scale])
test[num_cols_to_scale]  = scaler.transform(test[num_cols_to_scale])

print(f'Scaled {len(num_cols_to_scale)} numerical columns.')


Scaled 427 numerical columns.


## 10. Final Check

In [15]:
print('=== Final Dataset Summary ===')
print(f'Train      : {train.shape} | nulls: {train.isnull().sum().sum()}')
print(f'Validation : {val.shape}   | nulls: {val.isnull().sum().sum()}')
print(f'Test       : {test.shape}  | nulls: {test.isnull().sum().sum()}')
print(f'\nFraud rate — Train: {train[target].mean():.3%} | Val: {val[target].mean():.3%} | Test: {test[target].mean():.3%}')
print(f'\nNumerical features  : {len(num_cols_to_scale)}')
print(f'Label-enc features  : {len(encoders)}')

train.head()

=== Final Dataset Summary ===
Train      : (472443, 469) | nulls: 0
Validation : (29516, 469)   | nulls: 0
Test       : (88581, 469)  | nulls: 0

Fraud rate — Train: 3.499% | Val: 3.500% | Test: 3.498%

Numerical features  : 427
Label-enc features  : 41


,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,card3_email_freq,card4_addr1_freq,card4_email_freq,card5_addr1_freq,card5_email_freq,card6_addr1_freq,card6_email_freq,M_true_count,M_false_count,M_null_count
0,0,-1.284509,-0.604289,1,2413,184,41,3,103,2,...,-0.695481,0.030106,-0.640796,0.259196,-0.382951,-0.096293,-0.738902,-1.078728,-0.932218,1.332150
1,0,-1.324847,-0.456764,1,1853,69,41,3,103,1,...,-0.812649,0.012730,-0.543578,0.271113,-0.387638,-0.718234,-0.795902,-1.078728,-0.932218,1.332150
2,0,0.270974,-0.114800,4,10248,105,41,2,21,2,...,1.290954,-0.733461,0.226372,-0.768300,-0.651914,-0.548791,1.422227,1.496585,-0.296365,-0.738606
3,0,-0.191930,-0.061396,4,1779,219,41,2,76,2,...,0.103935,-0.810307,-0.493023,-0.767008,-0.726669,-0.639897,0.093006,1.496585,0.339488,-1.330251
4,0,-0.417743,0.463793,4,7539,0,41,3,103,2,...,1.290954,2.293398,1.540538,2.558713,1.911954,1.704828,1.422227,0.208928,2.247047,-1.330251


## 11. Save Outputs

In [16]:
save_dir = os.path.join('..', 'Data', 'Preprocessed')
os.makedirs(save_dir, exist_ok=True)

train.to_csv(os.path.join(save_dir, 'train.csv'), index=False)
val.to_csv(os.path.join(save_dir, 'val.csv'), index=False)
test.to_csv(os.path.join(save_dir, 'test.csv'), index=False)

with open(os.path.join(save_dir, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

with open(os.path.join(save_dir, 'encoders.pkl'), 'wb') as f:
    pickle.dump(encoders, f)

metadata = {
    'cat_cols': list(encoders.keys()),
    'num_cols': num_cols_to_scale,
    'vocab_sizes': vocab_sizes,
    'target': target
}

with open(os.path.join(save_dir, 'column_metadata.pkl'), 'wb') as f:
    pickle.dump(metadata, f)

print(f'Saved to ./{save_dir}/')
print('  train.csv, val.csv, test.csv')
print('  scaler.pkl, encoders.pkl, column_metadata.pkl')

Saved to ./..\Data\Preprocessed/
  train.csv, val.csv, test.csv
  scaler.pkl, encoders.pkl, column_metadata.pkl
